In [1]:
print(123)

123


In [2]:
"""
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget ${PREFIX}/01-agentic-rag/code/ingest.py
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py
"""

'\nPREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main\n\nwget ${PREFIX}/01-agentic-rag/code/ingest.py\nwget ${PREFIX}/01-agentic-rag/code/rag_helper.py\nwget ${PREFIX}/04-evaluation/code/evaluation_utils.py\n'

In [3]:

# load data 
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [5]:
# use only llm course data 
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [6]:
documents = documents_llm

In [7]:
# every document already has an id:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

# -> every record in the kb needs a unique identifier

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
# we want structured output and define structured data using pydantic:

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
# instructions for the llm: 
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
# initialize openai: 

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
# generate something that we can send to the llm it doesnt need to be json but we'll doit 
import json

user_prompt = json.dumps(doc)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
# instead of using response.output_text etc in order to get 
# structured output we do parse

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
response.output_parsed.questions

['Can I still join the course if I found it late?',
 'If I start the course now, is it still possible to get a certificate?',
 'Do I miss out on anything if I join after the course has already started?',
 'Is it okay to enroll now, or is it too late to join?',
 'What do I need to do to be eligible for the course certificate after joining late?']

In [15]:
# we just did this for 1 question from the faq, 
# we need to do this for all the 70+ items from the faq db 
# and we will reuse this multiple times so create utility functions 


# looks like our llm function from the first module, 
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join even though it already started?', 'If I join late, is it still possible to get the certificate?', 'Do I need to submit the project before submissions close to qualify for the certificate?', 'Can I take the course now and still be eligible for a certificate later?', 'What’s the deadline for project submission if I want the certificate?']


In [16]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=297)

In [ ]:
# hardcoded for this particular model and prices of the day when the video was recorded
from evaluation_utils import calc_price

calc_price(usage)

# price was 1/1000th of a cent 

{'input_cost': 0.00015525,
 'output_cost': 0.00040500000000000003,
 'total_cost': 0.00056025}

In [18]:
# this is the format that we want to obtain for the questions 

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still join even though it already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, is it still possible to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit the project before submissions close to qualify for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I take the course now and still be eligible for a certificate later?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want the certificate?',
  'document': '74eb249bbf'}]

In [19]:
# now we need to do this for all the items in the faq db 

import pandas as pd

In [ ]:
pd.DataFrame(records)

,question,document
0,I just found this course — can I still join ev...,74eb249bbf
1,"If I join late, is it still possible to get th...",74eb249bbf
2,Do I need to submit the project before submiss...,74eb249bbf
3,Can I take the course now and still be eligibl...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [21]:
# if something goes temporarily wrong e.g. due to a temporary or network issue, then retry

from evaluation_utils import llm_structured_retry

# we will se it in the processing function: 

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [22]:
# do it for all documents: 

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [23]:
# Parallel processing with 5 or 6 connections at once

# it is possible because we are just waiting for a response from openai
# we want to be a little carefult to not hit the provider limits somehow


# threadpoolexecutor allows us to have different workers, 
# we will have thread 1, thead 2, thread 3, ... each of these threads 
# will be an executor that is pulling of this common task pool. 

# then we iterate over this things, submit to the output executor, 
# thread pool executor is evaluating them and we get bakc the responses 

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# using it is very simple:
# we create this threat pool we say that we want at most 6 workders and then 
# we use this function

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(
        pool,
        documents,
        generate_ground_truth
    )

  0%|          | 0/118 [00:00<?, ?it/s]

In [24]:
results[0]

([{'question': 'Is it still possible to join the course if I found it late?',
   'document': '74eb249bbf'},
  {'question': 'Can I start the course after it has already begun?',
   'document': '74eb249bbf'},
  {'question': 'If I join the course now, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': "What do I need to do to qualify for the certificate if I'm joining late?",
   'document': '74eb249bbf'},
  {'question': 'Are project submissions only accepted during a certain period for the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=84, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=291))

In [25]:
# split data and usage into different lists: 


# extent (everything)
# append (just one)
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

# 9 cent 

0.09083999999999998

In [28]:
# price helper: 

from evaluation_utils import calc_total_price

calc_total_price(usages)

0.09083999999999998

In [29]:
# turn all this into a pandas document so we can look at it and 
# also download it 

import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth

,question,document
0,Is it still possible to join the course if I f...,74eb249bbf
1,Can I start the course after it has already be...,74eb249bbf
2,"If I join the course now, can I still get a ce...",74eb249bbf
3,What do I need to do to qualify for the certif...,74eb249bbf
4,Are project submissions only accepted during a...,74eb249bbf
...,...,...
585,Why am I getting 401 Unauthorized when I use m...,46efd1088d
586,Does an EU Logfire write token need a special ...,46efd1088d
587,What should I check first if Logfire seems to ...,46efd1088d
588,Could an old LOGFIRE_TOKEN in my shell be over...,46efd1088d


In [31]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [33]:
"""
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv

"""

'\nPREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main\n\nwget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv\n\n'

In [ ]:
"""
NOW WE HAVE THIS Qij*, Ai generated and we are sending this to the KB

WE GET BACK: documents d1, ... d5

WE NEED TO CHECK: IS THE RIGHT ANSWER A IN d1, .. d5
"""